# SHIFT UI API Workflow — End-to-End

This notebook demonstrates the full workflow available through the SHIFT UI API:

1. **Define a region** (polygon)
2. **Fetch parcels** from OpenStreetMap
3. **Cluster parcels** using area-aware, capacity-distance, or k-means strategies
4. **Build the distribution graph** (primary + secondary network)
5. **Visualize the result** on an interactive map

Each step mirrors what the SHIFT Distribution Network Studio UI does under the hood.

## Prerequisites

Start the UI API server before running this notebook:
```bash
python -m shift.ui_api
```
The server runs on `http://localhost:8000` by default.

In [1]:
import requests
import json

BASE_URL = "http://localhost:8000"


def api(path, payload=None):
    """Call the SHIFT UI API."""
    if payload is None:
        resp = requests.get(f"{BASE_URL}{path}")
    else:
        resp = requests.post(f"{BASE_URL}{path}", json=payload)
    if not resp.ok:
        # Show the actual error detail from the server
        try:
            detail = resp.json().get("detail", resp.text)
        except Exception:
            detail = resp.text
        raise RuntimeError(f"{resp.status_code} from {path}: {detail}")
    return resp.json()


# Verify server is running
health = api("/api/health")
print(f"Server status: {health['status']}")

Server status: ok


## 1. Define the Study Region

We define a polygon (in longitude/latitude) around a neighborhood in Fort Worth, TX.
This polygon acts as the service territory boundary for our synthetic feeder.

In [2]:
# Fort Worth, TX — derived from cached parcel dataset bounding box
polygon = [
    {"longitude": -97.3360, "latitude": 32.7503},
    {"longitude": -97.3298, "latitude": 32.7503},
    {"longitude": -97.3298, "latitude": 32.7560},
    {"longitude": -97.3360, "latitude": 32.7560},
]

# Source (substation) location — at the north edge of the polygon
source_location = {"longitude": -97.3330, "latitude": 32.7558}

print(f"Polygon: {len(polygon)} vertices")
print(f"Source: ({source_location['longitude']}, {source_location['latitude']})")

Polygon: 4 vertices
Source: (-97.333, 32.7558)


## 2. Fetch Parcels

Load building footprints from a cached dataset (originally fetched from OpenStreetMap).
If the API server is available and Overpass is reachable, live parcels are fetched instead.
Each parcel has a centroid, building type, and geometry.

In [3]:
from pathlib import Path
from collections import Counter

CACHE_FILE = Path("cached_parcels.json")

# Load parcels from cached dataset (originally fetched from OpenStreetMap).
# To fetch live data instead, uncomment the api() call below.
# parcel_data = api("/api/parcels/fetch", {"polygon": polygon})

parcel_data = json.load(CACHE_FILE.open())
print(f"Parcels loaded: {parcel_data['count']}")

# Show building type distribution
types = Counter(p["building_type"] for p in parcel_data["parcels"])
for btype, count in types.most_common():
    print(f"  {btype or 'unknown'}: {count}")

Parcels loaded: 61
  yes: 37
  commercial: 6
  parking: 6
  roof: 4
  apartments: 3
  garage: 1
  hotel: 1
  church: 1
  residential: 1
  office: 1


## 3. Cluster Parcels into Transformer Groups

SHIFT supports three clustering strategies:

| Strategy | Determines cluster count by |
|---|---|
| `kmeans_count` | Fixed user-specified count |
| `area_aware` | Total parcel area ÷ target area per transformer |
| `capacity_distance` | Estimated kVA loading + distance constraints |

Below we use **area_aware** clustering with:
- Target area per transformer: 54,000 sq ft (~5,016 m²)
- Dedicated transformer threshold: 22,000 sq ft (~2,044 m²) — parcels exceeding this get their own transformer

In [4]:
SQFT_TO_M2 = 0.09290304

cluster_payload = {
    "strategy": "area_aware",
    "parcels": parcel_data["parcels"],
    "points": [
        (p["geometry"][0] if isinstance(p["geometry"], list) else p["geometry"])
        for p in parcel_data["parcels"]
    ],
    "target_area_per_transformer_m2": 54000 * SQFT_TO_M2,  # 54,000 sq ft
    "dedicated_transformer_area_m2": 22000 * SQFT_TO_M2,  # 22,000 sq ft
    "num_clusters": 5,  # ignored by area_aware
}

cluster_data = api("/api/clusters/build", cluster_payload)

print(f"Strategy: {cluster_data['strategy']}")
print(f"Uses num_clusters: {cluster_data['uses_num_clusters']}")
print(f"Clusters created: {cluster_data['count']}")
print()

# Show diagnostics
details = cluster_data.get("strategy_details", {})
if details:
    print("Strategy diagnostics:")
    print(f"  Total parcels: {details.get('area_aware_total_parcels')}")
    print(f"  Dedicated transformers: {details.get('area_aware_dedicated_parcels')}")
    print(f"  Shared parcels: {details.get('area_aware_shared_parcels')}")
    print(f"  Shared area total: {details.get('area_aware_shared_area_total_m2', 0):.0f} m²")
    print(
        f"  Estimated shared clusters: {details.get('area_aware_estimated_shared_clusters_raw')}"
    )

print()
for i, c in enumerate(cluster_data["clusters"]):
    print(
        f"  Cluster {i + 1}: {c['num_points']} parcels @ ({c['center']['longitude']:.5f}, {c['center']['latitude']:.5f})"
    )

Strategy: area_aware
Uses num_clusters: False
Clusters created: 24

Strategy diagnostics:
  Total parcels: 61
  Dedicated transformers: 15
  Shared parcels: 46
  Shared area total: 42962 m²
  Estimated shared clusters: 9

  Cluster 1: 1 parcels @ (-97.33239, 32.75081)
  Cluster 2: 1 parcels @ (-97.33210, 32.75201)
  Cluster 3: 1 parcels @ (-97.33370, 32.75138)
  Cluster 4: 1 parcels @ (-97.33291, 32.75188)
  Cluster 5: 1 parcels @ (-97.33431, 32.75260)
  Cluster 6: 1 parcels @ (-97.33294, 32.75325)
  Cluster 7: 1 parcels @ (-97.33138, 32.75236)
  Cluster 8: 1 parcels @ (-97.33397, 32.75187)
  Cluster 9: 1 parcels @ (-97.33453, 32.75158)
  Cluster 10: 1 parcels @ (-97.33377, 32.75314)
  Cluster 11: 1 parcels @ (-97.33480, 32.75397)
  Cluster 12: 1 parcels @ (-97.33322, 32.75229)
  Cluster 13: 1 parcels @ (-97.33352, 32.75379)
  Cluster 14: 1 parcels @ (-97.33429, 32.75524)
  Cluster 15: 1 parcels @ (-97.33304, 32.75495)
  Cluster 16: 11 parcels @ (-97.33069, 32.75324)
  Cluster 17: 4 pa

## 4. Build the Distribution Graph

Build the primary + secondary network using the cluster groups.

Available secondary strategies:
- `OpenStreetSecondaryStrategy` — follows real roads (requires Overpass API)
- `DelaunayStrategy` — organic triangulation-based layout (no external API)
- `RadialStrategy` — direct star connections (simplest)

Below we use **Delaunay** for offline reliability. Switch to `OpenStreetSecondaryStrategy` when Overpass is available for road-following routing.

In [5]:
FT_TO_M = 0.3048

graph_payload = {
    "groups": [{"center": c["center"], "points": c["points"]} for c in cluster_data["clusters"]],
    "source_location": source_location,
    "polygon": polygon,
    "network_type": "balanced_default",
    "secondary_strategy": "DelaunayStrategy",
    "buffer_meters": 66 * FT_TO_M,
    "secondary_buffer_meters": 164 * FT_TO_M,
    "offline": True,  # geometric routing — no Overpass dependency
    # Set offline=False to use road-following routing (requires Overpass API)
}

graph_data = api("/api/graph/build", graph_payload)

summary = graph_data["summary"]
print("Graph built successfully!")
print(f"  Graph ID: {summary['graph_id']}")
print(f"  Nodes: {summary['node_count']}")
print(f"  Edges: {summary['edge_count']}")
print(
    f"  Total length: {summary['total_length_m']:.1f} m ({summary['total_length_m'] * 3.281:.0f} ft)"
)
print(f"  Transformers: {summary['transformer_hint_count']}")
print(f"  Load nodes: {summary['load_node_count']}")
print(f"  Routing: {summary['routing_strategy']}")
print(f"  Secondary: {summary['secondary_strategy']}")

Graph built successfully!
  Graph ID: graph-0041
  Nodes: 141
  Edges: 140
  Total length: 3415.6 m (11206 ft)
  Transformers: 24
  Load nodes: 61
  Routing: SteinerTreeStrategy
  Secondary: DelaunayStrategy


## 5. Visualize the Network on a Map

Plot the graph geometry (nodes and edges) on an interactive Folium map.
- **Blue lines**: Distribution branches (primary/secondary lines)
- **Red lines**: Transformer connections
- **Green markers**: Voltage source (substation)
- **Red markers**: Transformer nodes
- **Blue markers**: Load nodes

In [6]:
import folium

geometry = graph_data["geometry"]

# Center map on the source
center_lat = source_location["latitude"]
center_lon = source_location["longitude"]
m = folium.Map(location=[center_lat, center_lon], zoom_start=16)

# Draw polygon boundary
folium.Polygon(
    locations=[(p["latitude"], p["longitude"]) for p in polygon],
    color="#9a3412",
    weight=2,
    fill=True,
    fill_opacity=0.05,
    popup="Study Region",
).add_to(m)

# Draw edges
edge_colors = {
    "DistributionBranchBase": "#2563eb",
    "DistributionTransformer": "#dc2626",
}
for edge in geometry["edges"]:
    color = edge_colors.get(edge["type"], "#6b7280")
    weight = 3 if edge["type"] == "DistributionTransformer" else 2
    folium.PolyLine(
        locations=[
            [edge["from"]["latitude"], edge["from"]["longitude"]],
            [edge["to"]["latitude"], edge["to"]["longitude"]],
        ],
        color=color,
        weight=weight,
        opacity=0.85,
        popup=f"{edge['name']} ({edge['type']})",
    ).add_to(m)

# Draw nodes
for node in geometry["nodes"]:
    is_source = "DistributionVoltageSource" in node["assets"]
    is_transformer = node["name"].endswith("_ht")
    is_load = "DistributionLoad" in node["assets"]

    if is_source:
        color, radius = "#16a34a", 10
    elif is_transformer:
        color, radius = "#dc2626", 7
    elif is_load:
        color, radius = "#2563eb", 5
    else:
        color, radius = "#9ca3af", 3

    folium.CircleMarker(
        location=[node["location"]["latitude"], node["location"]["longitude"]],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=f"{node['name']}<br>{', '.join(node['assets']) if node['assets'] else 'bus'}",
    ).add_to(m)

# Draw parcel centroids
for p in parcel_data["parcels"]:
    geom = p["geometry"]
    pt = geom[0] if isinstance(geom, list) else geom
    folium.CircleMarker(
        location=[pt["latitude"], pt["longitude"]],
        radius=2,
        color="#0f766e",
        fill=True,
        fill_opacity=0.6,
    ).add_to(m)

m

## 6. Compare Routing Strategies

Run the same cluster groups through two network presets side-by-side to compare topology metrics.

In [7]:
# Use whatever secondary strategy succeeded in step 4
secondary_override = graph_payload.get("secondary_strategy")

compare_payload = {
    "builds": [
        {
            **graph_payload,
            "network_type": "balanced_default",
            "routing_strategy": None,
            "secondary_strategy": secondary_override,
        },
        {
            **graph_payload,
            "network_type": "road_optimized",
            "routing_strategy": None,
            "secondary_strategy": secondary_override,
        },
    ]
}

compare_data = api("/api/graph/compare", compare_payload)

print(f"{'Metric':<25} {'Balanced Default':<20} {'Road Optimized':<20}")
print("-" * 65)
for key in [
    "node_count",
    "edge_count",
    "total_length_m",
    "transformer_hint_count",
    "routing_strategy",
    "secondary_strategy",
]:
    v1 = compare_data["runs"][0].get(key, "—")
    v2 = compare_data["runs"][1].get(key, "—")
    if isinstance(v1, float):
        v1 = f"{v1:.1f}"
        v2 = f"{v2:.1f}"
    print(f"  {key:<23} {str(v1):<20} {str(v2):<20}")

Metric                    Balanced Default     Road Optimized      
-----------------------------------------------------------------
  node_count              141                  141                 
  edge_count              140                  140                 
  total_length_m          3415.6               3415.6              
  transformer_hint_count  24                   24                  
  routing_strategy        SteinerTreeStrategy  WeightedSteinerTreeStrategy
  secondary_strategy      DelaunayStrategy     DelaunayStrategy    


## 7. Experiment: Change Clustering Parameters

Demonstrate how different clustering thresholds affect transformer count.

In [8]:
experiments = [
    ("Large target (108k sqft)", 108000),
    ("Default (54k sqft)", 54000),
    ("Small target (27k sqft)", 27000),
]

print(f"{'Scenario':<30} {'Clusters':<10} {'Dedicated':<12} {'Shared':<10}")
print("-" * 62)

for label, target_sqft in experiments:
    payload = {
        "strategy": "area_aware",
        "parcels": parcel_data["parcels"],
        "points": cluster_payload["points"],
        "target_area_per_transformer_m2": target_sqft * SQFT_TO_M2,
        "dedicated_transformer_area_m2": 22000 * SQFT_TO_M2,
        "num_clusters": 5,
    }
    result = api("/api/clusters/build", payload)
    details = result.get("strategy_details", {})
    print(
        f"  {label:<28} {result['count']:<10} {details.get('area_aware_dedicated_parcels', '?'):<12} {details.get('area_aware_shared_parcels', '?'):<10}"
    )

Scenario                       Clusters   Dedicated    Shared    
--------------------------------------------------------------
  Large target (108k sqft)     20         15           46        
  Default (54k sqft)           24         15           46        
  Small target (27k sqft)      33         15           46        


## 8. Available API Endpoints

The SHIFT UI API exposes these endpoints:

| Endpoint | Method | Purpose |
|---|---|---|
| `/api/health` | GET | Server health check |
| `/api/options` | GET | Available strategies, types, and modes |
| `/api/parcels/fetch` | POST | Fetch building parcels from OSM |
| `/api/clusters/build` | POST | Cluster parcels into transformer groups |
| `/api/graph/build` | POST | Build primary + secondary distribution graph |
| `/api/graph/compare` | POST | Compare multiple graph builds side-by-side |
| `/api/feeders/auto-build` | POST | Automatically estimate feeder count and build |
| `/api/mapper/phase` | POST | Configure phase mapper |
| `/api/mapper/voltage` | POST | Configure voltage mapper |
| `/api/mapper/equipment` | POST | Configure equipment mapper |
| `/api/system/build` | POST | Build full `DistributionSystem` |
| `/api/system/export` | POST | Export system to JSON |

In [9]:
# Show all available options from the API
options = api("/api/options")
print(json.dumps(options, indent=2))

{
  "network_types": [
    "balanced_default",
    "road_optimized",
    "full_road_exploration"
  ],
  "cluster_strategies": [
    "kmeans_count",
    "area_aware",
    "capacity_distance"
  ],
  "cluster_balance_modes": [
    "balanced",
    "unbalanced"
  ],
  "routing_strategies": [
    "SteinerTreeStrategy",
    "WeightedSteinerTreeStrategy",
    "ShortestPathTreeStrategy",
    "MinimumSpanningTreeStrategy",
    "FullRoadGraphStrategy"
  ],
  "secondary_strategies": [
    "AutoDensitySecondaryStrategy",
    "MeshSteinerStrategy",
    "RadialStrategy",
    "DelaunayStrategy",
    "OpenStreetSecondaryStrategy",
    "HubLineStrategy"
  ],
  "phase_methods": [
    "agglomerative",
    "kmean",
    "greedy"
  ],
  "transformer_types": [
    "THREE_PHASE",
    "SINGLE_PHASE_PRIMARY_DELTA",
    "SINGLE_PHASE",
    "SPLIT_PHASE",
    "SPLIT_PHASE_PRIMARY_DELTA"
  ]
}


## 9. Configure Phase Mapper

Assign electrical phases to transformers. This determines the phase allocation
across the entire graph (nodes, branches, transformers).

Each transformer is configured with:
- **Type**: `THREE_PHASE`, `SINGLE_PHASE`, or `SPLIT_PHASE`
- **Capacity**: rated kVA

In [10]:
# Get the graph ID from the build step
graph_id = summary["graph_id"]

# List transformer edges to configure phase mapping
transformers_resp = api(f"/api/graph/{graph_id}/transformers")
transformer_names = transformers_resp["transformers"]
print(f"Transformers in graph: {len(transformer_names)}")

# Configure phase mapper — all transformers as THREE_PHASE, 500 kVA
phase_payload = {
    "graph_id": graph_id,
    "method": "greedy",
    "transformer_configs": [
        {"tr_name": name, "tr_type": "THREE_PHASE", "tr_capacity_kva": 500.0}
        for name in transformer_names
    ],
}

phase_result = api("/api/mapper/phase", phase_payload)
print(f"Phase mapper configured: {phase_result['count']} transformers")

Transformers in graph: 24
Phase mapper configured: 24 transformers


## 10. Configure Voltage Mapper

Set primary/secondary voltage levels across transformer boundaries.
This determines the voltage profile for the entire network.

- **Primary**: 12.47 kV (typical medium-voltage distribution)
- **Secondary**: 0.48 kV (480V, typical for commercial loads)

In [11]:
# Configure voltage mapper — 12.47 kV primary / 0.48 kV secondary
voltage_payload = {
    "graph_id": graph_id,
    "transformer_voltages": [
        {"name": name, "voltages_kv": [12.47, 0.48]} for name in transformer_names
    ],
}

voltage_result = api("/api/mapper/voltage", voltage_payload)
print(f"Voltage mapper configured: {voltage_result['count']} transformers")
print("  Primary: 12.47 kV, Secondary: 0.48 kV")

Voltage mapper configured: 24 transformers
  Primary: 12.47 kV, Secondary: 0.48 kV


## 11. Build the GDM Distribution System

With phase and voltage mappers configured via the API, we now use the SHIFT library
directly to:
1. Load a reference equipment catalog (`_build/model.json`)
2. Create an equipment mapper that assigns load + edge equipment from the catalog
3. Build the complete `DistributionSystem`

This step uses the library directly because the equipment mapper requires a
`node_asset_equipment_mapping` (load assignment) which is application-specific.

In [16]:
from pathlib import Path
from functools import cached_property
import importlib

from gdm.distribution import DistributionSystem
from gdm.distribution.components import DistributionTransformer, DistributionLoad
from gdm.distribution.equipment import LoadEquipment, PhaseLoadEquipment
from gdm.quantities import ActivePower, ReactivePower
from infrasys.component import Component

# Reload modules to pick up fixes
import shift.constants
import shift.system_builder

importlib.reload(shift.constants)
importlib.reload(shift.system_builder)

from shift.system_builder import DistributionSystemBuilder
from shift.mapper.edge_equipment_mapper import EdgeEquipmentMapper
from shift.data_model import VALID_NODE_TYPES

# Load catalog (as DistributionSystem — it contains equipment components)
catalog_path = Path("_build/model.json").resolve()
print(f"Loading catalog from: {catalog_path}")
catalog = DistributionSystem.from_json(catalog_path)
print(f"Catalog loaded: {catalog.name}")

# Re-build the graph locally using the same parameters (offline + Delaunay)
from shift import PRSG, GeoLocation
from shift.graph.secondary import DelaunayStrategy
from infrasys.quantities import Distance
from shift.data_model import GroupModel

groups = [
    GroupModel(
        center=GeoLocation(g["center"]["longitude"], g["center"]["latitude"]),
        points=[GeoLocation(p["longitude"], p["latitude"]) for p in g["points"]],
    )
    for g in graph_payload["groups"]
]
source = GeoLocation(source_location["longitude"], source_location["latitude"])

prsg_builder = PRSG(
    groups=groups,
    source_location=source,
    buffer=Distance(graph_payload["buffer_meters"], "m"),
    secondary_strategy=DelaunayStrategy(),
    offline=True,
)
graph = prsg_builder.get_distribution_graph()
print(f"Graph rebuilt: {len(list(graph.get_nodes()))} nodes, {len(list(graph.get_edges()))} edges")

Loading catalog from: /Users/alatif/Documents/GitHub/shift/docs/example/_build/model.json


2026-07-16 15:49:54.650 | INFO     | shift.graph.prsgb:build_primary_network:121 - Offline mode — using geometric primary network
2026-07-16 15:49:54.665 | DEBUG    | shift.graph.openstreet_graph_builder:get_distribution_graph:335 - Building secondary for GeoLocation(longitude=-97.33238687142857, latitude=32.75081207857143): 15477bd0-8f91-4aea-9ad5-c497590b754f
2026-07-16 15:49:54.666 | DEBUG    | shift.graph.openstreet_graph_builder:get_distribution_graph:335 - Building secondary for GeoLocation(longitude=-97.33210025, latitude=32.752012225): 04ac451b-eac1-4c7a-acc4-000c376a805e
2026-07-16 15:49:54.666 | DEBUG    | shift.graph.openstreet_graph_builder:get_distribution_graph:335 - Building secondary for GeoLocation(longitude=-97.33370305999999, latitude=32.751383520000005): f8e62eed-c08a-4a6a-9668-ee942f8d83db
2026-07-16 15:49:54.667 | DEBUG    | shift.graph.openstreet_graph_builder:get_distribution_graph:335 - Building secondary for GeoLocation(longitude=-97.33290653750001, latitude=3

Catalog loaded: None
Graph rebuilt: 141 nodes, 140 edges


## 12. Configure Mappers and Build System

Now apply phase mapping, voltage mapping, and equipment mapping using the SHIFT
library, then build the complete `DistributionSystem`.

In [17]:
from shift import (
    TransformerPhaseMapperModel,
    TransformerTypes,
    BalancedPhaseMapper,
    TransformerVoltageMapper,
    TransformerVoltageModel,
)
from gdm.distribution.components import DistributionVoltageSource
from gdm.distribution.equipment import (
    VoltageSourceEquipment,
    PhaseVoltageSourceEquipment,
    DistributionTransformerEquipment,
)
from gdm.distribution.enums import VoltageTypes
from gdm.quantities import ApparentPower, Voltage, Reactance
from infrasys.quantities import Resistance, Angle

# --- Check what transformer voltages are available in the catalog ---
catalog_xfmrs = list(catalog.get_components(DistributionTransformerEquipment))
if catalog_xfmrs:
    sample = catalog_xfmrs[0]
    wdg_voltages = [w.rated_voltage for w in sample.windings]
    print(f"Catalog transformer sample: {sample.name}")
    print(f"  Winding voltages: {wdg_voltages}")
    # Use the catalog's actual voltages
    primary_kv = max(wdg_voltages).to("kilovolt").magnitude
    secondary_kv = min(wdg_voltages).to("kilovolt").magnitude
    print(f"  Using: primary={primary_kv} kV, secondary={secondary_kv} kV")
else:
    primary_kv = 12.47
    secondary_kv = 0.48
    print(f"No transformers in catalog — using defaults: {primary_kv}/{secondary_kv} kV")

# --- Phase Mapper ---
phase_models = [
    TransformerPhaseMapperModel(
        tr_name=el.name,
        tr_type=TransformerTypes.THREE_PHASE,
        tr_capacity=ApparentPower(500, "kilova"),
        location=graph.get_node(from_node).location,
    )
    for from_node, _, el in graph.get_edges()
    if el.edge_type is DistributionTransformer
]
phase_mapper = BalancedPhaseMapper(graph, method="greedy", mapper=phase_models)
print(f"\nPhase mapper: {len(phase_models)} transformers configured")

# --- Voltage Mapper (using catalog-compatible voltages) ---
voltage_models = [
    TransformerVoltageModel(
        name=el.name,
        voltages=[Voltage(primary_kv, "kilovolt"), Voltage(secondary_kv, "kilovolt")],
    )
    for _, _, el in graph.get_edges()
    if el.edge_type is DistributionTransformer
]
voltage_mapper = TransformerVoltageMapper(graph, xfmr_voltage=voltage_models)
print(f"Voltage mapper: {len(voltage_models)} transformers @ {primary_kv}/{secondary_kv} kV")


# --- Equipment Mapper (with load + voltage source assignment) ---
class DefaultLoadEquipmentMapper(EdgeEquipmentMapper):
    """Equipment mapper that assigns default equipment to all asset nodes."""

    @cached_property
    def node_asset_equipment_mapping(self) -> dict[str, dict[VALID_NODE_TYPES, Component]]:
        mapping = {}

        # Default load equipment from catalog
        load_equipments = list(catalog.get_components(LoadEquipment))
        if load_equipments:
            default_load = load_equipments[0]
        else:
            default_load = LoadEquipment(
                name="default_load",
                phase_loads=[
                    PhaseLoadEquipment(
                        name="phase_a_load",
                        real_power=ActivePower(10, "kilowatt"),
                        reactive_power=ReactivePower(3, "kilovar"),
                        z_real=0.0,
                        z_imag=0.0,
                        i_real=0.0,
                        i_imag=0.0,
                        p_real=1.0,
                        p_imag=1.0,
                    )
                ],
            )

        # Default voltage source equipment
        vsource_equip = VoltageSourceEquipment(
            name="default_vsource",
            sources=[
                PhaseVoltageSourceEquipment(
                    name=f"vsrc_phase_{i}",
                    r0=Resistance(0.001, "ohm"),
                    r1=Resistance(0.001, "ohm"),
                    x0=Reactance(0.001, "ohm"),
                    x1=Reactance(0.001, "ohm"),
                    voltage=Voltage(primary_kv, "kilovolt"),
                    voltage_type=VoltageTypes.LINE_TO_LINE,
                    angle=Angle(i * 120, "degree"),
                )
                for i in range(3)
            ],
        )

        for node in graph.get_nodes():
            if not node.assets:
                continue
            node_mapping = {}
            if DistributionLoad in node.assets:
                node_mapping[DistributionLoad] = default_load
            if DistributionVoltageSource in node.assets:
                node_mapping[DistributionVoltageSource] = vsource_equip
            if node_mapping:
                mapping[node.name] = node_mapping
        return mapping


equipment_mapper = DefaultLoadEquipmentMapper(
    graph=graph,
    catalog_sys=catalog,
    voltage_mapper=voltage_mapper,
    phase_mapper=phase_mapper,
)
print("\nEquipment mapper: ready")
print(
    f"  Load nodes: {sum(1 for v in equipment_mapper.node_asset_equipment_mapping.values() if DistributionLoad in v)}"
)
print(
    f"  VSource nodes: {sum(1 for v in equipment_mapper.node_asset_equipment_mapping.values() if DistributionVoltageSource in v)}"
)

# --- Build the System ---
sys_builder = DistributionSystemBuilder(
    name="fort_worth_feeder",
    dist_graph=graph,
    phase_mapper=phase_mapper,
    voltage_mapper=voltage_mapper,
    equipment_mapper=equipment_mapper,
)

system = sys_builder.get_system()
print("\n✓ GDM DistributionSystem built successfully!")
system.info()

Catalog transformer sample: 00013351-aea3-4cfa-8daf-a47bd27f4ffa
  Winding voltages: [<Quantity(7.19976905, 'kilovolt')>, <Quantity(0.277136259, 'kilovolt')>]
  Using: primary=7.199769053117783 kV, secondary=0.27713625866050806 kV

Phase mapper: 24 transformers configured
Voltage mapper: 24 transformers @ 7.199769053117783/0.27713625866050806 kV

Equipment mapper: ready
  Load nodes: 61
  VSource nodes: 1

✓ GDM DistributionSystem built successfully!


System                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Property                         ┃             Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ System name                      │ fort_worth_feeder │
│ Data format version              │             2.3.3 │
│ Components attached              │               472 │
│ Time Series attached             │                 0 │
│ Supplemental Attributes attached │                 0 │
│ Description                      │                   │
└──────────────────────────────────┴───────────────────┘

Component Information                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Type                             ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ DistributionBus                  │   141 │
│ DistributionLoad                 │    61 │
│ DistributionTransformer          │    24 │
│ DistributionTransformerEquipment │     1 │
│ DistributionVoltageSource        │     1 │
│ LoadEquipment                    │     1 │
│ Location                         │   117 │
│ MatrixImpedanceBranch            │   116 │
│ MatrixImpedanceBranchEquipment   │     1 │
│ PhaseLoadEquipment               │     3 │
│ PhaseVoltageSourceEquipment      │     3 │
│ VoltageSourceEquipment           │     1 │
│ WindingEquipment                 │     2 │
└──────────────────────────────────┴───────┘

## 13. Export the GDM Model

Export the built system to JSON. This file is a complete GDM `DistributionSystem`
that can be loaded by any GDM-compatible tool for power flow analysis.

In [18]:
# Export to JSON
output_dir = Path("_build")
output_dir.mkdir(exist_ok=True)
export_path = output_dir / "fort_worth_feeder.json"

system.to_json(str(export_path), overwrite=True)
print(f"Exported to: {export_path}")
print(f"File size: {export_path.stat().st_size / 1024:.1f} KB")

# Verify reload
from gdm.distribution import DistributionSystem

reloaded = DistributionSystem.from_json(str(export_path))
print(f"\n✓ Reload verified — {reloaded.name}")
reloaded.info()

Exported to: _build/fort_worth_feeder.json
File size: 307.9 KB

✓ Reload verified — fort_worth_feeder


System                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Property                         ┃             Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ System name                      │ fort_worth_feeder │
│ Data format version              │             2.3.3 │
│ Components attached              │               472 │
│ Time Series attached             │                 0 │
│ Supplemental Attributes attached │                 0 │
│ Description                      │                   │
└──────────────────────────────────┴───────────────────┘

Component Information                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Type                             ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ DistributionBus                  │   141 │
│ DistributionLoad                 │    61 │
│ DistributionTransformer          │    24 │
│ DistributionTransformerEquipment │     1 │
│ DistributionVoltageSource        │     1 │
│ LoadEquipment                    │     1 │
│ Location                         │   117 │
│ MatrixImpedanceBranch            │   116 │
│ MatrixImpedanceBranchEquipment   │     1 │
│ PhaseLoadEquipment               │     3 │
│ PhaseVoltageSourceEquipment      │     3 │
│ VoltageSourceEquipment           │     1 │
│ WindingEquipment                 │     2 │
└──────────────────────────────────┴───────┘